<a href="https://colab.research.google.com/github/jagadeeshdandu/NASSCOM-AI-FDP/blob/main/Day_4_Data_Cleaning_%26_Preprocessing_(Part_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.2


In [2]:
# -----------------------------------------------------------
# A SMALL, ALREADY-CLEAN DATASET (Part 1 did the cleaning)
# -----------------------------------------------------------
# Mixed columns: an unordered category (city), an ordered category
# (size), two numeric features on very different scales, and a target.
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})
df

,city,size,age,income,bought
0,pune,small,25,38000,0
1,delhi,large,41,92000,1
2,mumbai,medium,33,55000,0
3,pune,medium,29,47000,0
4,delhi,small,52,120000,1
5,mumbai,large,38,76000,1
6,pune,large,46,88000,1
7,delhi,small,22,41000,0


In [3]:
# -----------------------------------------------------------
# 🔹 1A. ONE-HOT ENCODING (for UNORDERED categories)
# -----------------------------------------------------------

# 'city' has no natural order -> one 0/1 column per category
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe

,city_delhi,city_mumbai,city_pune
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0
6,0,0,1
7,1,0,0


In [4]:
# -----------------------------------------------------------
# 🔹 1B. LABEL / ORDINAL ENCODING (for ORDERED categories)
# -----------------------------------------------------------

# 'size' has a real order: small < medium < large -> map to 0,1,2
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]

,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


In [7]:
# one-hot encode just the 'city' column of the whole df

df_city_encoded = pd.get_dummies(df['city'], prefix='city').astype(int)
display(df_city_encoded)

,city_delhi,city_mumbai,city_pune
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0
6,0,0,1
7,1,0,0


### Why one-hot encoding is generally wrong for 'size' (ordinal data):

*   **'Size' is ordinal:** The 'size' column has an inherent order: 'small' < 'medium' < 'large'.
*   **One-hot encoding ignores order:** One-hot encoding creates separate binary columns for each category (e.g., `size_small`, `size_medium`, `size_large`). This implies that these categories are independent and have no relational order, which is false for 'size'.
*   **Increased dimensionality & sparse data:** For categories with many unique values, one-hot encoding can lead to a very high number of columns, increasing the dimensionality of the dataset and potentially making it sparser, which can negatively impact some models.
*   **Misleading to models:** By treating ordered categories as unordered, machine learning models might not correctly interpret the relationship between them, leading to less accurate predictions.

### Why ordinal encoding is generally wrong for 'city' (nominal data):

*   **'City' is nominal:** The 'city' column (e.g., 'pune', 'delhi', 'mumbai') has no inherent order. There's no logical reason to say 'delhi' < 'mumbai' < 'pune'.
*   **Ordinal encoding imposes arbitrary order:** If you were to apply ordinal encoding to 'city', you'd assign arbitrary numerical values (e.g., Pune=0, Delhi=1, Mumbai=2). This would create a false sense of order or magnitude that doesn't exist in reality.
*   **Misleading to models:** Machine learning models would then interpret these numerical values as having an ordered relationship, potentially giving more weight or an incorrect relationship to cities assigned higher numbers. This can lead to biased or incorrect model learning. For instance, a model might incorrectly assume that 'Mumbai' (2) is 'greater' or 'more important' than 'Pune' (0) just because of its assigned numerical value.

In [8]:
# -----------------------------------------------------------
# 🔹 2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES
# -----------------------------------------------------------

# income (tens of thousands) dwarfs age (tens). A distance-based
# model would treat income as nearly all that matters.
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])

        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0


In [9]:
# -----------------------------------------------------------
# 🔹 2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)        # mean 0, std 1
m = MinMaxScaler().fit_transform(num)          # range [0, 1]

print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))

Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


In [10]:
income = df[['income']]   # 2D shape for sklearn

In [11]:
# standardise -> check mean ~0, std ~1

df_standardized = pd.DataFrame(z, columns=['age', 'income'])
print('Mean of standardized data:')
display(df_standardized.mean().round(2))
print('\nStandard deviation of standardized data:')
display(df_standardized.std().round(2))

Mean of standardized data:


,0
age,0.0
income,-0.0



Standard deviation of standardized data:


,0
age,1.07
income,1.07


In [12]:
# min-max scale -> check min 0, max 1

df_min_max_scaled = pd.DataFrame(m, columns=['age', 'income'])
print('Minimum of Min-Max scaled data:')
display(df_min_max_scaled.min().round(2))
print('\nMaximum of Min-Max scaled data:')
display(df_min_max_scaled.max().round(2))

Minimum of Min-Max scaled data:


,0
age,0.0
income,0.0



Maximum of Min-Max scaled data:


,0
age,1.0
income,1.0


## When to Scale Features (and when not to)

### Features that generally **need scaling**:

*   **Numerical features with widely different ranges or units**: As seen with 'age' (tens) and 'income' (tens of thousands). Many machine learning algorithms (especially distance-based ones like K-Nearest Neighbors, SVMs, or algorithms that use gradient descent like neural networks) are sensitive to the scale of input features.
    *   **Examples from our data**: `age`, `income`
*   **Features with skewed distributions**: Scaling can sometimes help normalize these distributions, which can be beneficial for some models.

### Features that generally **do not need scaling**:

*   **Binary features**: These are already in a fixed range (0 or 1). Scaling them would not change their meaning or their relationship to other features in a way that benefits most algorithms.
*   **One-hot encoded categorical features**: Similar to binary features, these are already represented as 0s and 1s and do not have an inherent 'scale' that would impact distance calculations in a problematic way.
    *   **Example from our data**: The `city` columns after one-hot encoding (`city_delhi`, `city_mumbai`, `city_pune`)
*   **Ordinal encoded features, if the model can interpret the order**: If a model explicitly understands and utilizes the ordinal relationship (e.g., decision trees, random forests), scaling might not be strictly necessary, although it's often still applied to ensure consistency with other scaled numerical features.
    *   **Example from our data**: `size_code` (0, 1, 2) – while it has a range, the relative difference between 0 and 1 is as important as between 1 and 2.
*   **Features for tree-based models**: Decision trees and random forests are generally not sensitive to feature scaling because they split data based on thresholds, not distances. The splits remain the same regardless of the feature's scale.

### Key takeaway:

Scaling is crucial for algorithms that rely on distance metrics or gradient descent. For other algorithms, it might not be strictly necessary but can still be a good practice for consistency and sometimes for minor performance benefits.

In [13]:
# -----------------------------------------------------------
# 🔹 3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT + transform on train
X_test_s  = scaler.transform(X_test)        # only TRANSFORM on test
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))

train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


In [14]:
# 1a. WRONG: fit on ALL data, then split (leakage!)
wrong_mean = StandardScaler().fit(X).mean_

In [15]:
# 1b. RIGHT: fit on TRAIN only
right_mean = StandardScaler().fit(X_train).mean_

In [16]:
print('fit-on-all mean :', wrong_mean.round(1))
print('fit-on-train mean:', right_mean.round(1))

fit-on-all mean : [3.5800e+01 6.9625e+04]
fit-on-train mean: [3.45e+01 6.90e+04]


## Why fitting on all data (before splitting) is a problem: Data Leakage

### The Problem: Data Leakage

When you fit a scaler (like `StandardScaler` or `MinMaxScaler`) on the *entire* dataset (`X`) before splitting it into training and testing sets, you are inadvertently allowing information from the test set to "leak" into the training process. This is called **data leakage**.

### How it happens:

*   **Scaler learns from the future**: The scaler calculates parameters (like mean and standard deviation for `StandardScaler`, or min and max for `MinMaxScaler`) using *all* the data, including the data points that will later be used for testing.
*   **Test set is no longer pristine**: When you then `transform` the test set, it's being transformed based on statistics (mean, std, min, max) that were influenced by the test set itself. This means your test set is no longer a truly unseen and independent dataset.

### Why it's detrimental:

1.  **Over-optimistic performance estimates**: Your model will appear to perform better on the test set than it would in a real-world scenario where it encounters truly new, unseen data. This is because the test data has, in a subtle way, already contributed to shaping the scaling parameters applied to it.
2.  **Poor generalization**: A model trained with data leakage will not generalize well to new, genuinely unseen data because its perceived good performance during evaluation was based on an illusion.
3.  **Misleading model selection**: If you're comparing different models or hyperparameters, data leakage can lead you to choose a model that looks good on your

In [17]:
# -----------------------------------------------------------
# 🔹 4A. COMBINE & BIN EXISTING COLUMNS
# -----------------------------------------------------------

fe = df.copy()
# combine: income per year of age (a crude 'earning rate')
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)
# bin: turn continuous age into life-stage buckets
fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]


,age,income,income_per_age,age_group
0,25,38000,1520.0,young
1,41,92000,2244.0,mid
2,33,55000,1667.0,mid
3,29,47000,1621.0,young
4,52,120000,2308.0,senior
5,38,76000,2000.0,mid
6,46,88000,1913.0,senior
7,22,41000,1864.0,young


In [18]:
# -----------------------------------------------------------
# 🔹 4B. EXTRACT FROM A DATE
# -----------------------------------------------------------

dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()   # Monday, Tuesday, ...
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d

,date,day_of_week,month,is_weekend
0,2024-01-06,Saturday,1,True
1,2024-03-15,Friday,3,False
2,2024-07-21,Sunday,7,True
3,2024-12-25,Wednesday,12,False


In [19]:
ex = df.copy()

In [20]:
# derive: a 'high_earner' flag (income > median)
median_income = fe['income'].median()
fe['high_earner'] = (fe['income'] > median_income).astype(int)
fe[['income', 'high_earner']]

,income,high_earner
0,38000,0
1,92000,1
2,55000,0
3,47000,0
4,120000,1
5,76000,1
6,88000,1
7,41000,0


In [22]:
# bin income into 3 buckets with pd.cut

fe['income_group'] = pd.cut(fe['income'], bins=3, labels=['low', 'medium', 'high'])
display(fe[['income', 'high_earner', 'income_group']])

,income,high_earner,income_group
0,38000,0,low
1,92000,1,medium
2,55000,0,low
3,47000,0,low
4,120000,1,high
5,76000,1,medium
6,88000,1,medium
7,41000,0,low


In [23]:
# -----------------------------------------------------------
# 🔹 5A. ONE LEAK-FREE PIPELINE: PREPROCESS + MODEL
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']

# scale the numbers, one-hot the categories — all in one object
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])


In [24]:
# -----------------------------------------------------------
# 🔹 5B. FIT THE WHOLE THING IN ONE CALL
# -----------------------------------------------------------

X = df[num_cols + cat_cols]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

pipe.fit(X_train, y_train)        # preprocessing fitted on TRAIN only — no leakage
acc = pipe.score(X_test, y_test)  # transforms test with train-fitted steps
print('Test accuracy:', round(acc, 2))
print('(small toy dataset — the point is the leak-free workflow, not the score)')

Test accuracy: 1.0
(small toy dataset — the point is the leak-free workflow, not the score)


In [25]:
from sklearn.preprocessing import MinMaxScaler

Xn = df[['age', 'income']]
yn = df['bought']

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

num_cols = ['age', 'income']
cat_cols = ['city', 'size']

# Create a ColumnTransformer for preprocessing
preprocessor_minmax = ColumnTransformer([
    ('num', MinMaxScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# Create the pipeline with MinMaxScaler and LogisticRegression
pipe_minmax_lr = Pipeline([
    ('preprocessor', preprocessor_minmax),
    ('classifier', LogisticRegression(max_iter=1000))
])

print(pipe_minmax_lr)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', MinMaxScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('classifier', LogisticRegression(max_iter=1000))])


In [27]:
# Pipeline: MinMaxScaler -> LogisticRegression

X = df[num_cols + cat_cols]
y = df['bought']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

# Fit the pipeline on the training data
pipe_minmax_lr.fit(X_train, y_train)

# Evaluate the pipeline on the test data
acc_minmax = pipe_minmax_lr.score(X_test, y_test)
print('Test accuracy with MinMaxScaler pipeline:', round(acc_minmax, 2))

Test accuracy with MinMaxScaler pipeline: 1.0
